In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import hashlib
import sys
from datetime import datetime

# Project folders
PROJECT = Path.cwd()

RAW = PROJECT / "data" / "raw"
PROCESSED = PROJECT / "data" / "processed"
REPORTS = PROJECT / "reports"
DELIVERABLES = PROJECT / "deliverables"

for folder in [RAW, PROCESSED, REPORTS, DELIVERABLES]:
    folder.mkdir(parents=True, exist_ok=True)

print("Python version:", sys.version)
print("Project folder:", PROJECT)
print("Raw data folder:", RAW)
print("Processed data folder:", PROCESSED)
print("Reports folder:", REPORTS)

In [ ]:
%pip install pandas numpy openpyxl matplotlib seaborn scikit-learn

In [7]:
files = sorted(RAW.glob("*"))

for file in files:
    if file.is_file():
        print(f"{file.name:45} {file.stat().st_size / 1024**2:.2f} MB")

01 Introduction.xlsx                          0.03 MB
02 Field Production.xlsx                      14.85 MB
03 Train Cases 1.xlsx                         315.86 MB
04 Train Cases 2.xlsx                         315.92 MB
05 Train Cases 3.xlsx                         315.89 MB
06 Train Cases 4.xlsx                         157.98 MB
07 Validation Cases.xlsx                      236.90 MB
08 Test Cases.xlsx                            236.92 MB
09 Template Deliverable.xlsx                  0.03 MB


In [ ]:
excel_files = list(RAW.glob("*.xlsx")) + list(RAW.glob("*.xlsm"))

for file in excel_files:
    print(f"\n{'=' * 70}")
    print("WORKBOOK:", file.name)

    workbook = pd.ExcelFile(file)

    for sheet in workbook.sheet_names:
        preview = pd.read_excel(
            file,
            sheet_name=sheet,
            nrows=5
        )

        print(f"\nSheet: {sheet}")
        print("Columns:", list(preview.columns))
        print("Preview shape:", preview.shape)
        display(preview.head(3))

In [ ]:
from openpyxl import load_workbook

inventory = []

for file in excel_files:
    wb = load_workbook(
        file,
        read_only=True,
        data_only=True
    )

    for ws in wb.worksheets:
        inventory.append({
            "workbook": file.name,
            "sheet": ws.title,
            "rows": ws.max_row,
            "columns": ws.max_column
        })

    wb.close()

inventory_df = pd.DataFrame(inventory)

display(inventory_df)

inventory_df.to_csv(
    REPORTS / "workbook_inventory.csv",
    index=False
)

In [6]:
simulation_file = next(
    (p for p in RAW.glob("*.xlsx")
     if "train" in p.name.lower()),
    None
)

if simulation_file is None:
    print("Select your actual simulation workbook.")
else:
    print(simulation_file)

c:\Users\iffah\Downloads\dataworks\data\raw\03 Train Cases 1.xlsx


In [ ]:
import pandas as pd

# Store information about every worksheet
workbook_report = []

for file in excel_files:
    print(f"Inspecting: {file.name}")

    try:
        workbook = pd.ExcelFile(file)

        for sheet in workbook.sheet_names:
            try:
                df = pd.read_excel(
                    file,
                    sheet_name=sheet
                )

                numeric_df = df.select_dtypes(include="number")

                placeholder_count = int(
                    numeric_df.eq(-999.25).sum().sum()
                )

                workbook_report.append({
                    "workbook": file.name,
                    "sheet": sheet,
                    "rows": len(df),
                    "columns": len(df.columns),
                    "missing_values": int(
                        df.isna().sum().sum()
                    ),
                    "duplicate_rows": int(
                        df.duplicated().sum()
                    ),
                    "placeholder_-999.25": placeholder_count
                })

            except Exception as e:
                print(f"  Could not read sheet '{sheet}': {e}")

    except Exception as e:
        print(f"  Could not open workbook: {e}")

# Display the inspection results
workbook_report_df = pd.DataFrame(workbook_report)

display(workbook_report_df)

In [ ]:
# Display all rows and columns
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

display(workbook_report_df)

In [13]:
# Define where to save the report
report_path = REPORTS / "workbook_inspection_report.csv"

# Save the report as a CSV file
workbook_report_df.to_csv(
    report_path,
    index=False
)

print("Inspection report saved successfully!")
print(report_path)

Inspection report saved successfully!
c:\Users\iffah\Downloads\dataworks\reports\workbook_inspection_report.csv


In [ ]:
from pathlib import Path
import pandas as pd

# Locate the raw workbooks
RAW = Path.cwd() / "data" / "raw"

files_to_inspect = {
    "01 Introduction.xlsx": [
        "Nomenclature",
        "Uncertainty Values"
    ],
    "02 Field Production.xlsx": [
        "Field Production",
        "Gas production cumulative",
        "Oil production rate",
        "Oil production cumulative"
    ],
    "03 Train Cases 1.xlsx": ["Case 1"],
    "04 Train Cases 2.xlsx": ["Case 21"],
    "05 Train Cases 3.xlsx": ["Case 41"],
    "06 Train Cases 4.xlsx": ["Case 61"],
    "07 Validation Cases.xlsx": ["Case 71"],
    "08 Test Cases.xlsx": ["Case 86"],
    "09 Template Deliverable.xlsx": ["Case 1"]
}

for filename, sheets in files_to_inspect.items():
    filepath = RAW / filename

    print("\n" + "=" * 70)
    print("WORKBOOK:", filename)

    for sheet in sheets:
        df = pd.read_excel(
            filepath,
            sheet_name=sheet,
            nrows=5
        )

        print(f"\nSHEET: {sheet}")
        print("Columns:", len(df.columns))
        print("Column names:", df.columns.tolist())

        display(df.head(3))

In [ ]:


import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

RAW = Path.cwd() / "data" / "raw"

# Sheets we want to inspect
targets = {
    "01 Introduction.xlsx": ["Nomenclature", "Uncertainty Values"],
    "02 Field Production.xlsx": [
        "Field Production",
        "Gas production cumulative",
        "Oil production rate",
        "Oil production cumulative"
    ],
    "03 Train Cases 1.xlsx": ["Case 1"]
}

output = []

for filename, sheets in targets.items():
    # Find the workbook using its filename
    matches = [
        p for p in RAW.glob("*.xlsx")
        if filename.lower().replace(".xlsx", "") in p.stem.lower()
    ]

    if not matches:
        output.append(f"\n### WORKBOOK NOT FOUND: {filename}")
        continue

    file = matches[0]

    for sheet in sheets:
        output.append(f"\n{'=' * 70}")
        output.append(f"WORKBOOK: {file.name} | SHEET: {sheet}")
        output.append("=" * 70)

        try:
            df = pd.read_excel(file, sheet_name=sheet)

            output.append(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
            output.append(f"Column names: {df.columns.tolist()}")
            output.append("\nFirst 5 rows:")
            output.append(df.head(5).to_string(index=False))
            output.append("\nLast 3 rows:")
            output.append(df.tail(3).to_string(index=False))

        except Exception as e:
            output.append(f"Could not read sheet: {e}")

# Display everything as ONE output block
result_text = "\n".join(output)

print(result_text)

Profile Case 1

In [ ]:

import pandas as pd
from pathlib import Path

RAW = Path.cwd() / "data" / "raw"
file = next(
    p for p in RAW.glob("*.xlsx")
    if "03 Train Cases 1" in p.stem
)

df = pd.read_excel(file, sheet_name="Case 1")

# Count placeholder values in each column
placeholder_counts = df.eq(-999.25).sum()

# Count actual blank cells in each column
missing_counts = df.isna().sum()

# Summarize the data quality
summary = pd.DataFrame({
    "Data type": df.dtypes.astype(str),
    "Missing values": missing_counts,
    "Placeholder -999.25": placeholder_counts,
    "Zero values": df.eq(0).sum(),
    "Unique values": df.nunique()
})

print("Case 1 dimensions:", df.shape)
print("\nData quality summary:")
display(summary)

print("\nRows with PORV = 0:",
      (df["Pore Volume (PORV)"] == 0).sum())

print("Rows with at least one -999.25:",
      df.eq(-999.25).any(axis=1).sum())

print("Rows with all 29 values equal to -999.25:",
      df.eq(-999.25).all(axis=1).sum())

In [21]:

# Check the relationship between zero pore volume and placeholders

zero_porv = df["Pore Volume (PORV)"] == 0
has_placeholder = df.eq(-999.25).any(axis=1)

print("Total rows:", len(df))

print("\nRows with zero PORV AND a placeholder:",
      (zero_porv & has_placeholder).sum())

print("Rows with zero PORV but NO placeholder:",
      (zero_porv & ~has_placeholder).sum())

print("Rows with positive PORV AND a placeholder:",
      ((df["Pore Volume (PORV)"] > 0) & has_placeholder).sum())

print("Rows with negative PORV:",
      (df["Pore Volume (PORV)"] < 0).sum())

print("\nPORV summary:")
display(df["Pore Volume (PORV)"].describe())

Total rows: 90365

Rows with zero PORV AND a placeholder: 47853
Rows with zero PORV but NO placeholder: 0
Rows with positive PORV AND a placeholder: 0
Rows with negative PORV: 0

PORV summary:


count    9.036500e+04
mean     4.704588e+04
std      2.572445e+05
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      1.742356e+04
max      2.426922e+06
Name: Pore Volume (PORV), dtype: float64

In [ ]:

# Count placeholder values in each column,
# separately for zero-PORV and positive-PORV cells

zero_porv = df["Pore Volume (PORV)"] == 0
positive_porv = df["Pore Volume (PORV)"] > 0

placeholder_check = pd.DataFrame({
    "Placeholder in zero-PORV cells":
        df.loc[zero_porv].eq(-999.25).sum(),
    "Placeholder in positive-PORV cells":
        df.loc[positive_porv].eq(-999.25).sum(),
    "Total placeholder values":
        df.eq(-999.25).sum()
})

display(placeholder_check)

In [ ]:

# Compare reservoir properties for zero- and positive-PORV cells

zero_porv = df["Pore Volume (PORV)"] == 0
positive_porv = df["Pore Volume (PORV)"] > 0

properties = [
    "Porosity (PORO)",
    "Permeability I (PERMX)",
    "Permeability J (PERMY)",
    "Permeability K (PERMZ)",
    "Pressure (PRESSURE)",
    "Water Saturation (SWAT)",
    "Oil Saturation (SOIL)",
    "Gas Saturation (SGAS)"
]

rows = []

for col in properties:
    rows.append({
        "Property": col,
        "Zero-PORV placeholder count":
            df.loc[zero_porv, col].eq(-999.25).sum(),
        "Positive-PORV placeholder count":
            df.loc[positive_porv, col].eq(-999.25).sum(),
        "Positive-PORV minimum":
            df.loc[positive_porv, col].min(),
        "Positive-PORV maximum":
            df.loc[positive_porv, col].max()
    })

display(pd.DataFrame(rows).round(4))

Check Saturation Consistency

In [25]:

# Check fluid saturation consistency in positive-PORV cells

positive_cells = df[df["Pore Volume (PORV)"] > 0].copy()

positive_cells["Saturation Sum"] = (
    positive_cells["Water Saturation (SWAT)"]
    + positive_cells["Oil Saturation (SOIL)"]
    + positive_cells["Gas Saturation (SGAS)"]
)

print("Saturation sum statistics:")
display(positive_cells["Saturation Sum"].describe())

print("\nRows where saturation sum differs from 1 by more than 0.01:")
saturation_issues = positive_cells[
    (positive_cells["Saturation Sum"] - 1).abs() > 0.01
]

print("Count:", len(saturation_issues))
display(
    saturation_issues[
        [
            "I Index", "J Index", "K Index",
            "Water Saturation (SWAT)",
            "Oil Saturation (SOIL)",
            "Gas Saturation (SGAS)",
            "Saturation Sum"
        ]
    ].head(10)
)

Saturation sum statistics:


count    42512.0
mean         1.0
std          0.0
min          1.0
25%          1.0
50%          1.0
75%          1.0
max          1.0
Name: Saturation Sum, dtype: float64


Rows where saturation sum differs from 1 by more than 0.01:
Count: 0


,I Index,J Index,K Index,Water Saturation (SWAT),Oil Saturation (SOIL),Gas Saturation (SGAS),Saturation Sum


Profile All 70 Training Cases

In [26]:

from pathlib import Path
import pandas as pd

RAW = Path.cwd() / "data" / "raw"
REPORTS = Path.cwd() / "reports"
REPORTS.mkdir(parents=True, exist_ok=True)

marker = -999.25
results = []

# Read all four training workbooks
train_files = sorted(
    p for p in RAW.glob("*.xlsx")
    if "train cases" in p.stem.lower()
)

for file in train_files:
    xl = pd.ExcelFile(file)

    for sheet in xl.sheet_names:
        if not sheet.lower().startswith("case"):
            continue

        df_case = pd.read_excel(file, sheet_name=sheet)

        porv = df_case["Pore Volume (PORV)"]
        zero_porv = porv.eq(0)
        positive_porv = porv.gt(0)

        saturation_cols = [
            "Water Saturation (SWAT)",
            "Oil Saturation (SOIL)",
            "Gas Saturation (SGAS)"
        ]

        # Exclude placeholder-containing rows from saturation validation
        valid_saturation = (
            positive_porv
            & ~df_case[saturation_cols].eq(marker).any(axis=1)
            & df_case[saturation_cols].notna().all(axis=1)
        )

        saturation_sum = df_case.loc[
            valid_saturation, saturation_cols
        ].sum(axis=1)

        results.append({
            "workbook": file.name,
            "case": sheet,
            "rows": len(df_case),
            "columns": len(df_case.columns),
            "missing_cells": int(df_case.isna().sum().sum()),
            "placeholder_cells": int(df_case.eq(marker).sum().sum()),
            "rows_with_placeholder": int(
                df_case.eq(marker).any(axis=1).sum()
            ),
            "zero_PORV_rows": int(zero_porv.sum()),
            "positive_PORV_rows": int(positive_porv.sum()),
            "negative_PORV_rows": int(porv.lt(0).sum()),
            "saturation_rows_checked": int(valid_saturation.sum()),
            "saturation_sum_issues": int(
                ((saturation_sum - 1).abs() > 0.01).sum()
            )
        })

report = pd.DataFrame(results)

output_file = REPORTS / "training_cases_quality_profile.csv"
report.to_csv(output_file, index=False)

print("Cases profiled:", len(report))
print("Report saved to:", output_file)
display(report)

Cases profiled: 70
Report saved to: c:\Users\iffah\Downloads\dataworks\reports\training_cases_quality_profile.csv


,workbook,case,rows,columns,missing_cells,placeholder_cells,rows_with_placeholder,zero_PORV_rows,positive_PORV_rows,negative_PORV_rows,saturation_rows_checked,saturation_sum_issues
0,03 Train Cases 1.xlsx,Case 1,90365,29,0,1056951,47853,47853,42512,0,42512,0
1,03 Train Cases 1.xlsx,Case 2,90365,29,0,1056951,47853,47853,42512,0,42512,0
2,03 Train Cases 1.xlsx,Case 3,90365,29,0,1056951,47853,47853,42512,0,42512,0
3,03 Train Cases 1.xlsx,Case 4,90365,29,0,1056951,47853,47853,42512,0,42512,0
4,03 Train Cases 1.xlsx,Case 5,90365,29,0,1056951,47853,47853,42512,0,42512,0
5,03 Train Cases 1.xlsx,Case 6,90365,29,0,1056951,47853,47853,42512,0,42512,0
6,03 Train Cases 1.xlsx,Case 7,90365,29,0,1056951,47853,47853,42512,0,42512,0
7,03 Train Cases 1.xlsx,Case 8,90365,29,0,1056951,47853,47853,42512,0,42512,0
8,03 Train Cases 1.xlsx,Case 9,90365,29,0,1056951,47853,47853,42512,0,42512,0
9,03 Train Cases 1.xlsx,Case 10,90365,29,0,1056951,47853,47853,42512,0,42512,0


Baca nomenclature

In [ ]:

from pathlib import Path
import pandas as pd

RAW = Path.cwd() / "data" / "raw"

intro_file = next(
    p for p in RAW.glob("*.xlsx")
    if "01 Introduction" in p.stem
)

nomenclature = pd.read_excel(
    intro_file,
    sheet_name="Nomenclature",
    header=None
)

print(nomenclature.to_string(index=False, header=False))


In [29]:
import pandas as pd
from pathlib import Path

# Locate the original Case 1 workbook
RAW = Path.cwd() / "data" / "raw"
REPORTS = Path.cwd() / "reports"
PROCESSED = Path.cwd() / "data" / "processed"

PROCESSED.mkdir(parents=True, exist_ok=True)

file = RAW / "03 Train Cases 1.xlsx"

# Load Case 1 without changing the original file
df = pd.read_excel(file, sheet_name="Case 1")

# Make a separate working copy
cleaned_df = df.copy()

print("Original shape:", df.shape)
print("Working copy shape:", cleaned_df.shape)
print("Original file preserved:", file.exists())

Original shape: (90365, 29)
Working copy shape: (90365, 29)
Original file preserved: True


Check Proposed Cleaning Rule 

In [30]:
marker = -999.25

# Identify cells with zero pore volume
zero_pore_volume = cleaned_df["Pore Volume (PORV)"].eq(0)

# Identify reservoir-property columns, excluding coordinates,
# grid indices and pore volume
property_columns = [
    col for col in cleaned_df.columns
    if col not in [
        "I Index",
        "J Index",
        "K Index",
        "X Coordinate",
        "Y Coordinate",
        "Z Coordinate",
        "Pore Volume (PORV)"
    ]
]

# Count placeholder values in zero-PORV cells
placeholder_count = (
    cleaned_df.loc[zero_pore_volume, property_columns]
    .eq(marker)
    .sum()
    .sum()
)

print("Zero-PORV rows:", zero_pore_volume.sum())
print("Placeholder values in their reservoir properties:",
      placeholder_count)

# Verify whether positive-PORV rows contain placeholders
positive_pore_volume = cleaned_df["Pore Volume (PORV)"].gt(0)

positive_placeholder_count = (
    cleaned_df.loc[positive_pore_volume, property_columns]
    .eq(marker)
    .sum()
    .sum()
)

print("Placeholders in positive-PORV rows:",
      positive_placeholder_count)

Zero-PORV rows: 47853
Placeholder values in their reservoir properties: 1052766
Placeholders in positive-PORV rows: 0


Check Coordinate Values
(Some coordinates have -999.25 [unusable])

In [31]:
marker = -999.25

coordinate_columns = [
    "X Coordinate",
    "Y Coordinate",
    "Z Coordinate"
]

# Count placeholder values in each coordinate column
print("Placeholder counts in coordinates:")
print(df[coordinate_columns].eq(marker).sum())

# Count rows where all three coordinates are valid
valid_coordinates = ~df[coordinate_columns].eq(marker).any(axis=1)

print("\nRows with all three valid coordinates:",
      valid_coordinates.sum())

# Compare coordinate validity with pore volume
print("\nCoordinate validity by pore volume:")
print(
    df.assign(Valid_Coordinates=valid_coordinates)
      .groupby("Pore Volume (PORV)")["Valid_Coordinates"]
      .agg(["count", "sum"])
      .head()
)

Placeholder counts in coordinates:
X Coordinate    1395
Y Coordinate    1395
Z Coordinate    1395
dtype: int64

Rows with all three valid coordinates: 88970

Coordinate validity by pore volume:
                    count    sum
Pore Volume (PORV)              
0.000000            47853  46458
0.629475                1      1
3.893268                1      1
5.804942                1      1
5.933069                1      1


Cleaning Summary

In [32]:
# Summarise Case 1 before cleaning

marker = -999.25

summary = pd.DataFrame({
    "Check": [
        "Total rows",
        "Total columns",
        "Zero-PORV rows",
        "Positive-PORV rows",
        "Rows with invalid coordinates",
        "Rows with valid coordinates",
        "Rows with placeholders in property columns"
    ],
    "Count": [
        len(cleaned_df),
        len(cleaned_df.columns),
        cleaned_df["Pore Volume (PORV)"].eq(0).sum(),
        cleaned_df["Pore Volume (PORV)"].gt(0).sum(),
        cleaned_df[coordinate_columns].eq(marker).any(axis=1).sum(),
        (~cleaned_df[coordinate_columns].eq(marker).any(axis=1)).sum(),
        (
            cleaned_df[property_columns].eq(marker)
            .any(axis=1)
            .sum()
        )
    ]
})

display(summary)

,Check,Count
0,Total rows,90365
1,Total columns,29
2,Zero-PORV rows,47853
3,Positive-PORV rows,42512
4,Rows with invalid coordinates,1395
5,Rows with valid coordinates,88970
6,Rows with placeholders in property columns,47853


Apply Cleaning Rule

In [33]:
import numpy as np
from pathlib import Path

# Create a separate copy
cleaned_df = df.copy()

marker = -999.25

# Identify inactive cells
zero_pore_volume = cleaned_df["Pore Volume (PORV)"].eq(0)

# Replace placeholders only in reservoir-property columns
# and only for zero-PORV rows.
cleaning_log = []

for col in property_columns:
    mask = zero_pore_volume & cleaned_df[col].eq(marker)
    changed = int(mask.sum())

    cleaned_df.loc[mask, col] = np.nan

    if changed > 0:
        cleaning_log.append({
            "Column": col,
            "Values converted to missing": changed
        })

cleaning_log_df = pd.DataFrame(cleaning_log)

print("Original shape:", df.shape)
print("Processed shape:", cleaned_df.shape)
print("Placeholder values converted:",
      cleaning_log_df["Values converted to missing"].sum())
print("Remaining placeholders in property columns:",
      cleaned_df[property_columns].eq(marker).sum().sum())

display(cleaning_log_df)

Original shape: (90365, 29)
Processed shape: (90365, 29)
Placeholder values converted: 1052766
Remaining placeholders in property columns: 0


,Column,Values converted to missing
0,Porosity (PORO),47853
1,Permeability I (PERMX),47853
2,Permeability J (PERMY),47853
3,Permeability K (PERMZ),47853
4,Net-to-Gross Ratio (NTG),47853
5,Transmissibility I (TRANX),47853
6,Transmissibility J (TRANY),47853
7,Transmissibility K (TRANZ),47853
8,Fluid In Place Region (FIPNUM),47853
9,PVT Region (PVTNUM),47853


In [34]:
print("Original marker count:",
      df[property_columns].eq(-999.25).sum().sum())

print("Processed missing values in property columns:",
      cleaned_df[property_columns].isna().sum().sum())

print("Original shape unchanged:", df.shape == (90365, 29))
print("Processed shape preserved:", cleaned_df.shape == df.shape)

Original marker count: 1052766
Processed missing values in property columns: 1052766
Original shape unchanged: True
Processed shape preserved: True


In [35]:
# STEP 5: Validate the processed Case 1 data

marker = -999.25

# 1. Check dimensions
print("Original shape:", df.shape)
print("Processed shape:", cleaned_df.shape)

# 2. Check remaining placeholders in property columns
print(
    "Remaining property placeholders:",
    cleaned_df[property_columns].eq(marker).sum().sum()
)

# 3. Check missing values in property columns
print(
    "Missing property values after cleaning:",
    cleaned_df[property_columns].isna().sum().sum()
)

# 4. Check that all grid indices are unchanged
index_columns = ["I Index", "J Index", "K Index"]

print(
    "Grid indices unchanged:",
    df[index_columns].equals(cleaned_df[index_columns])
)

# 5. Check positive-PORV rows remain unchanged
positive_mask = df["Pore Volume (PORV)"].gt(0)

print(
    "Positive-PORV rows unchanged:",
    df.loc[positive_mask].equals(cleaned_df.loc[positive_mask])
)

Original shape: (90365, 29)
Processed shape: (90365, 29)
Remaining property placeholders: 0
Missing property values after cleaning: 1052766
Grid indices unchanged: True
Positive-PORV rows unchanged: True


In [5]:
from pathlib import Path
from openpyxl import load_workbook
from collections import Counter

RAW = Path.cwd() / "data" / "raw"
file = RAW / "03 Train Cases 1.xlsx"

wb = load_workbook(
    file,
    read_only=True,
    data_only=True
)

try:
    ws = wb["Case 1"]

    rows = ws.iter_rows(values_only=True)
    headers = next(rows)

    counts = Counter()

    for row in rows:
        for col, value in zip(headers, row):
            if isinstance(value, (int, float)) and value == -999.25:
                counts[col] += 1

    print("Parameters containing -999.25:")
    for col, count in counts.most_common():
        print(f"{col}: {count:,} occurrences")

finally:
    wb.close()

Parameters containing -999.25:
Porosity (PORO): 47,853 occurrences
Permeability I (PERMX): 47,853 occurrences
Permeability J (PERMY): 47,853 occurrences
Permeability K (PERMZ): 47,853 occurrences
Net-to-Gross Ratio (NTG): 47,853 occurrences
Transmissibility I (TRANX): 47,853 occurrences
Transmissibility J (TRANY): 47,853 occurrences
Transmissibility K (TRANZ): 47,853 occurrences
Fluid In Place Region (FIPNUM): 47,853 occurrences
PVT Region (PVTNUM): 47,853 occurrences
Connate Water Saturation (SWL): 47,853 occurrences
Critical Water Saturation (SWCR): 47,853 occurrences
Maximum Water Saturation (SWU): 47,853 occurrences
 Connateg  Gas Saturation (SGL): 47,853 occurrences
Critical Gas Saturation (SGCR): 47,853 occurrences
Residual Oil to Water (SOWCR): 47,853 occurrences
Pressure (PRESSURE): 47,853 occurrences
Water Saturation (SWAT): 47,853 occurrences
Oil Saturation (SOIL): 47,853 occurrences
Gas Saturation (SGAS): 47,853 occurrences
Solution Gas-Oil Ratio (RS): 47,853 occurrences
Vap

In [4]:
#What parameters involved with data -999.25?
from pathlib import Path
from openpyxl import load_workbook
from collections import Counter
import pandas as pd

RAW = Path.cwd() / "data" / "raw"
REPORTS = Path.cwd() / "reports"
REPORTS.mkdir(parents=True, exist_ok=True)

MARKER = -999.25

files = [
    "03 Train Cases 1.xlsx",
    "04 Train Cases 2.xlsx",
    "05 Train Cases 3.xlsx",
    "06 Train Cases 4.xlsx",
    "07 Validation Cases.xlsx",
    "08 Test Cases.xlsx",
]

totals = Counter()
details = []

for filename in files:
    path = RAW / filename
    if not path.exists():
        print(f"Not found: {filename}")
        continue

    print(f"Scanning {filename}...", flush=True)

    wb = load_workbook(
        path, read_only=True, data_only=True
    )

    try:
        for ws in wb.worksheets:
            if not ws.title.strip().lower().startswith("case "):
                continue

            # Read header once
            rows = ws.iter_rows(values_only=True)
            headers = next(rows, None)

            if not headers:
                continue

            counts = Counter()

            for row in rows:
                for i, value in enumerate(row):
                    if (
                        isinstance(value, (int, float))
                        and value == MARKER
                        and i < len(headers)
                    ):
                        counts[str(headers[i])] += 1

            for column, count in counts.items():
                totals[column] += count
                details.append({
                    "workbook": filename,
                    "case": ws.title,
                    "column": column,
                    "marker_count": count
                })

    finally:
        wb.close()

summary = pd.DataFrame(
    [
        {"column": col, "total_marker_count": count}
        for col, count in totals.most_common()
    ]
)

detail_df = pd.DataFrame(details)

display(summary)

summary.to_csv(
    REPORTS / "placeholder_column_summary.csv",
    index=False
)

detail_df.to_csv(
    REPORTS / "placeholder_columns_by_case.csv",
    index=False
)

print("Reports saved in:", REPORTS)

Scanning 03 Train Cases 1.xlsx...


Exception ignored while calling deallocator <function ZipFile.__del__ at 0x000002151CD0DBC0>:
Traceback (most recent call last):
  File "c:\Users\iffah\AppData\Local\Programs\Python\Python314\Lib\zipfile\__init__.py", line 2046, in __del__
    self.close()
  File "c:\Users\iffah\AppData\Local\Programs\Python\Python314\Lib\zipfile\__init__.py", line 2063, in close
    self.fp.seek(self.start_dir)
ValueError: seek of closed file


KeyboardInterrupt: 

Clean Case 1

In [37]:
saved_df = pd.read_excel(output_file)

print("Saved file shape:", saved_df.shape)
print(
    "Remaining property placeholders:",
    saved_df[property_columns].eq(-999.25).sum().sum()
)
print(
    "Missing property values:",
    saved_df[property_columns].isna().sum().sum()
)

Saved file shape: (90365, 29)
Remaining property placeholders: 0
Missing property values: 1052766


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

RAW = Path.cwd() / "data" / "raw"
PROCESSED = Path.cwd() / "data" / "processed"
REPORTS = Path.cwd() / "reports"

PROCESSED.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

MARKER = -999.25
PORV = "Pore Volume (PORV)"

files = [
    "03 Train Cases 1.xlsx",
    "04 Train Cases 2.xlsx",
    "05 Train Cases 3.xlsx",
    "06 Train Cases 4.xlsx",
    "07 Validation Cases.xlsx",
    "08 Test Cases.xlsx",
]

summary = []

for filename in files:
    src = RAW / filename

    if not src.exists():
        print(f"SKIPPED: {filename}")
        continue

    out = PROCESSED / f"{src.stem}_Cleaned.xlsx"
    print(f"\nProcessing {filename}...", flush=True)

    with pd.ExcelWriter(out, engine="openpyxl") as writer:
        xls = pd.ExcelFile(src)

        for sheet in xls.sheet_names:
            # Pass the source filename, NOT the open workbook object.
            df = pd.read_excel(src, sheet_name=sheet)

            if not sheet.strip().lower().startswith("case "):
                df.to_excel(
                    writer, sheet_name=sheet[:31], index=False
                )
                continue

            if PORV not in df.columns:
                raise ValueError(
                    f"{filename} / {sheet}: PORV column missing"
                )

            excluded = [
                "I Index", "J Index", "K Index",
                "X Coordinate", "Y Coordinate", "Z Coordinate",
                PORV
            ]

            props = [c for c in df.columns if c not in excluded]

            delete_mask = (
                df[PORV].eq(0)
                & df[props].eq(MARKER).any(axis=1)
            )

            before = len(df)
            deleted = int(delete_mask.sum())

            df_clean = df.loc[~delete_mask].copy()

            df_clean.to_excel(
                writer, sheet_name=sheet[:31], index=False
            )

            summary.append({
                "workbook": filename,
                "case": sheet,
                "rows_before": before,
                "rows_deleted": deleted,
                "rows_after": len(df_clean),
            })

            print(
                f"  {sheet}: deleted {deleted:,} rows",
                flush=True
            )

        xls.close()

    print(f"Saved: {out}", flush=True)

summary_df = pd.DataFrame(summary)

summary_df.to_csv(
    REPORTS / "row_deletion_report.csv",
    index=False
)

display(summary_df)

print("\nAll processing finished.")
print("Cleaned files:", PROCESSED)
print("Report:", REPORTS / "row_deletion_report.csv")



Processing 03 Train Cases 1.xlsx...
  Case 1: deleted 47,853 rows
  Case 2: deleted 47,853 rows
  Case 3: deleted 47,853 rows
  Case 4: deleted 47,853 rows
  Case 5: deleted 47,853 rows
  Case 6: deleted 47,853 rows
  Case 7: deleted 47,853 rows
  Case 8: deleted 47,853 rows
  Case 9: deleted 47,853 rows
  Case 10: deleted 47,853 rows
  Case 11: deleted 47,853 rows
  Case 12: deleted 47,853 rows
  Case 13: deleted 47,853 rows
  Case 14: deleted 47,853 rows
  Case 15: deleted 47,853 rows
  Case 16: deleted 47,853 rows
  Case 17: deleted 47,853 rows
  Case 18: deleted 47,853 rows
  Case 19: deleted 47,853 rows
  Case 20: deleted 47,853 rows
Saved: c:\Users\iffah\Downloads\dataworks\data\processed\03 Train Cases 1_Cleaned.xlsx

Processing 04 Train Cases 2.xlsx...
  Case 21: deleted 47,853 rows
  Case 22: deleted 47,853 rows
  Case 23: deleted 47,853 rows
  Case 24: deleted 47,853 rows
  Case 25: deleted 47,853 rows
  Case 26: deleted 47,853 rows
  Case 27: deleted 47,853 rows
  Case 28: 

,workbook,case,rows_before,rows_deleted,rows_after
0,03 Train Cases 1.xlsx,Case 1,90365,47853,42512
1,03 Train Cases 1.xlsx,Case 2,90365,47853,42512
2,03 Train Cases 1.xlsx,Case 3,90365,47853,42512
3,03 Train Cases 1.xlsx,Case 4,90365,47853,42512
4,03 Train Cases 1.xlsx,Case 5,90365,47853,42512
...,...,...,...,...,...
95,08 Test Cases.xlsx,Case 96,90365,47853,42512
96,08 Test Cases.xlsx,Case 97,90365,47853,42512
97,08 Test Cases.xlsx,Case 98,90365,47853,42512
98,08 Test Cases.xlsx,Case 99,90365,47853,42512



All processing finished.
Cleaned files: c:\Users\iffah\Downloads\dataworks\data\processed
Report: c:\Users\iffah\Downloads\dataworks\reports\row_deletion_report.csv


DATA QUALITY ASSESSMENT